In [268]:
import torch
import pandas as pd
import numpy as np 
from torch import nn
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
from pathlib import Path

In [269]:
train_path = Path.cwd() / "data/train.csv"
test_path = Path.cwd() / "data/test.csv"
submission_path = Path.cwd() / "data/sample_submission.csv"


In [270]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)


print(train.head())
print(train['health_condition'].value_counts())



   id health_condition  sleep_duration  heart_rate    bmi  \
0   0        unhealthy            5.22        70.6  25.66   
1   1          at-risk            5.53        71.3  25.84   
2   2        unhealthy            5.29        75.4  24.54   
3   3        unhealthy            4.70        77.2  23.13   
4   4          at-risk            7.23        73.4  28.44   

   calorie_expenditure  step_count  exercise_duration  water_intake diet_type  \
0               2174.0      1326.0               19.8          1.86       veg   
1               1966.0      9891.0               49.9          1.26   non-veg   
2               2688.0     14216.0               38.1          1.60       veg   
3               2630.0      7174.0               59.9          2.02       veg   
4               2560.0      6584.0               46.0          2.25       veg   

  stress_level sleep_quality physical_activity_level smoking_alcohol  gender  
0         high       average               sedentary             ye

In [271]:
he = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
le = LabelEncoder()
scaler = MinMaxScaler()

def normalise(df : pd.DataFrame, train_data : bool = True):

    df_X = df.drop(["health_condition", "id"], axis=1)
    df_y = df["health_condition"]
    
    string_df = df_X.select_dtypes("object").columns.to_list()
    if train_data:
        encoded_str = he.fit_transform(df_X[string_df])
        normal_y = np.array(le.fit_transform(df_y), dtype=np.float32)
    else:
        encoded_str = he.transform(df_X[string_df])
        normal_y = np.array(le.transform(df_y), dtype=np.float32)

    str_normal = pd.DataFrame(
        encoded_str,
        columns=he.get_feature_names_out(string_df),
        index=df_X.index
    )
    str_normal = str_normal.drop(columns=[c for c in str_normal.columns if c.endswith("_nan")])

    float_df = df_X.select_dtypes("float64").columns.to_list()
    if train_data:

        encoded_float = scaler.fit_transform(df_X[float_df])
    else:
        encoded_float = scaler.transform(df_X[float_df])

    float_normal = pd.DataFrame(
        encoded_float,
        columns=scaler.get_feature_names_out(float_df),
        index=df_X.index
    )

    normal_X = pd.concat([float_normal, str_normal], axis=1)

    normal_X.fillna(normal_X.median(numeric_only=True), inplace=True)
    
    normal_X = normal_X.to_numpy(dtype=np.float32)
    

    return normal_X, normal_y

In [272]:
import xgboost as xgb
import lightgbm as lgb
import catboost as ctb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
print(train["health_condition"].value_counts())
train_df, temp_df = train_test_split(train, test_size=0.3, random_state=42, stratify=train['health_condition'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['health_condition'])

X_train, y_train = normalise(train_df, train_data=True)
X_val, y_val = normalise(val_df, train_data=False)
X_test, y_test = normalise(test_df, train_data=False)

model_XGB = xgb.XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss"
)

model_LGB = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
)

model_CTB = ctb.CatBoostClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    bootstrap_type="Bernoulli",
    subsample=0.8,
    colsample_bylevel=0.8,
    eval_metric="MultiClass"
)

class_weights = compute_sample_weight(class_weight='balanced', y=y_train)



health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64


In [273]:
print(len(X_train), len(y_train))
model_XGB.fit(X_train, y_train, sample_weight=class_weights)
model_LGB.fit(X_train, y_train, sample_weight=class_weights)
model_CTB.fit(X_train, y_train, sample_weight=class_weights)

483061 483061
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001809 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1820
[LightGBM] [Info] Number of data points in the train set: 483061, number of used features: 25
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

CatBoostClassifier(bootstrap_type='Bernoulli', colsample_bylevel=0.8, eval_metric='MultiClass', learning_rate=0.05, max_depth=6, n_estimators=600, subsample=0.8)

In [274]:
from sklearn.metrics import accuracy_score
y_pred_XGB = model_XGB.predict(X_val)
y_pred_LGB = model_LGB.predict(X_val)
y_pred_CTB = model_CTB.predict(X_val)

acc_XGB = accuracy_score(y_val, y_pred_XGB)
acc_LGB = accuracy_score(y_val, y_pred_LGB)
acc_CTB = accuracy_score(y_val, y_pred_CTB)
print("Accuracy")
print(f"XGB: {acc_XGB}, LGB: {acc_LGB}, CTB: {acc_CTB}")

report_XGB = classification_report(y_val, y_pred_XGB)
report_LGB = classification_report(y_val, y_pred_LGB)
report_CTB = classification_report(y_val, y_pred_CTB)

print("REPORT")
print(report_XGB)
print(report_LGB)
print(report_CTB)

print("Confusion Matrix")
print(confusion_matrix(y_val, y_pred_XGB))
print(confusion_matrix(y_val, y_pred_LGB))
print(confusion_matrix(y_val, y_pred_CTB))

/Users/almazbaktygali/Desktop/ML/ml_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy
XGB: 0.9409542762744776, LGB: 0.9416788229497745, CTB: 0.9389255455836465
REPORT
              precision    recall  f1-score   support

         0.0       0.99      0.94      0.97     88884
         1.0       0.74      0.95      0.83      5971
         2.0       0.70      0.96      0.81      8658

    accuracy                           0.94    103513
   macro avg       0.81      0.95      0.87    103513
weighted avg       0.95      0.94      0.94    103513

              precision    recall  f1-score   support

         0.0       0.99      0.94      0.97     88884
         1.0       0.74      0.95      0.83      5971
         2.0       0.71      0.96      0.81      8658

    accuracy                           0.94    103513
   macro avg       0.81      0.95      0.87    103513
weighted avg       0.95      0.94      0.95    103513

              precision    recall  f1-score   support

         0.0       0.99      0.94      0.96     88884
         1.0       0.73      0.95      

In [275]:
# y_test_pred = model.predict(X_test)
# acc_test = accuracy_score(y_test, y_test_pred)
# print(acc_test)
# report = classification_report(y_test, y_test_pred)
# print(report)


# print(confusion_matrix(y_test, y_test_pred))


proba_xgb = model_XGB.predict_proba(X_val)
proba_lgb = model_LGB.predict_proba(X_val)
proba_cgb = model_CTB.predict_proba(X_val)


blended_proba = (proba_xgb + proba_lgb + proba_cgb)/3
y_pred_blend = blended_proba.argmax(axis=1)
print(classification_report(y_val, y_pred_blend, target_names=le.classes_))

/Users/almazbaktygali/Desktop/ML/ml_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

     at-risk       0.99      0.94      0.97     88884
         fit       0.74      0.95      0.83      5971
   unhealthy       0.70      0.96      0.81      8658

    accuracy                           0.94    103513
   macro avg       0.81      0.95      0.87    103513
weighted avg       0.95      0.94      0.94    103513



In [276]:
def nonlabel_normalise(df : pd.DataFrame):
    id = df["id"].to_list()
    df_X = df.drop("id", axis=1)
    string_df = df_X.select_dtypes("object").columns.to_list()
    encoded_str = he.fit_transform(df_X[string_df])
    encoded_str = he.transform(df_X[string_df])

    str_normal = pd.DataFrame(
        encoded_str,
        columns=he.get_feature_names_out(string_df),
        index=df_X.index
    )
    str_normal = str_normal.drop(columns=[c for c in str_normal.columns if c.endswith("_nan")])

    float_df = df_X.select_dtypes("float64").columns.to_list()

    encoded_float = scaler.fit_transform(df_X[float_df])

    encoded_float = scaler.transform(df_X[float_df])

    float_normal = pd.DataFrame(
        encoded_float,
        columns=scaler.get_feature_names_out(float_df),
        index=df_X.index
    )

    normal_X = pd.concat([float_normal, str_normal], axis=1)
    normal_X.fillna(df.median(numeric_only=True), inplace=True)
    normal_X = normal_X.to_numpy(dtype=np.float32)
    

    return id, normal_X

In [277]:
id, X_test = nonlabel_normalise(test)
proba_xgb_test = model_XGB.predict_proba(X_test)
proba_lgb_test = model_LGB.predict_proba(X_test)
proba_ctb_test = model_CTB.predict_proba(X_test)

blended_proba_test = (proba_xgb_test + proba_lgb_test + proba_ctb_test)/3
y_pred_blended_test = blended_proba_test.argmax(axis=1)
submission = pd.DataFrame(
    {"id": id,
     "health_condition": le.inverse_transform(y_pred_blended_test)
    }
)

/Users/almazbaktygali/Desktop/ML/ml_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [278]:
print(submission)

            id health_condition
0       690088        unhealthy
1       690089          at-risk
2       690090          at-risk
3       690091          at-risk
4       690092        unhealthy
...        ...              ...
295748  985836              fit
295749  985837          at-risk
295750  985838          at-risk
295751  985839          at-risk
295752  985840        unhealthy

[295753 rows x 2 columns]


In [279]:
submission.to_csv(submission_path, index_label=False)